# IPL Win Probability & Score Prediction -- Merged Pipeline

**Default entry point** for `merged_project`. Runs project_gagan's pipeline end-to-end: Elo/form/H2H feature engineering, pre-match and dynamic 2nd/1st-innings win-probability models, phase-specific evaluation, the score regression zoo, the locked-2026 external holdout evaluation, SHAP explainability, and the 2026 tournament simulation.

All the underlying logic lives in `../src/` (tested by `../tests/`, run `pytest ../tests/` to verify) -- this notebook is a thin, narrated runner over that library, in the same style as `project_gagan/notebooks/ipl_pipeline_2008_2026.ipynb`.

See `../README.md` for what came from which source project and `../docs/known_limitations.md` for the honest caveats behind every number below.

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '..')   # resolve src/ from notebooks/

import pandas as pd

from src.pipeline import (
    load_and_prepare, train_pre_match_internal, build_dynamic_2nd,
    build_dynamic_1st, train_dynamic_internal, phase_specific_eval,
    train_score_zoo_and_save, retrain_pre_match_full, evaluate_2026_pre_match,
    PRE_FEAT,
)
from src.explainability import shap_importance
from src.tournament import actual_points_table, predicted_points_table, compare_tables

DATA_XLSX = '../data/raw/ipl_data.xlsx'
PM26_CSV = '../data/external_2026/prematch_dataset.csv'
IG26_CSV = '../data/external_2026/ingame_dataset.csv'
SCORE_PIPELINE_PATH = '../models/ipl_score_pipeline.pkl'

## 2. Load Data & Compute Elo/Form/H2H Features

Ported from `project_gagan/src/data.py` (`load_ball_by_ball`, `build_match_table`) and `src/elo.py` / `src/features.py`.

In [2]:
print('Loading data and computing Elo/form/H2H features...')
df, match_df = load_and_prepare(DATA_XLSX)
print(f'  Matches: {len(match_df)}  Deliveries: {len(df):,}')
match_df.head()

Loading data and computing Elo/form/H2H features...


  Matches: 1146  Deliveries: 273,503


,match_id,team1,team2,winner,year,venue,toss_winner,toss_decision,score1,score2,...,toss_bat_first,toss_field_first,elo1,elo2,elo_diff,form1,form2,h2h,form_diff,h2h_beta
0,335982,Kolkata Knight Riders,Royal Challengers Bengaluru,Kolkata Knight Riders,2008,M Chinnaswamy Stadium,Royal Challengers Bengaluru,field,222,82,...,0,1,1500.0,1500.0,0.0,0.5,0.5,0.5,0.0,0.5
1,335983,Chennai Super Kings,Punjab Kings,Chennai Super Kings,2008,"Punjab Cricket Association Stadium, Mohali",Chennai Super Kings,bat,240,207,...,1,0,1500.0,1500.0,0.0,0.5,0.5,0.5,0.0,0.5
2,335984,Rajasthan Royals,Delhi Capitals,Delhi Capitals,2008,Feroz Shah Kotla,Rajasthan Royals,bat,129,132,...,1,0,1500.0,1500.0,0.0,0.5,0.5,0.5,0.0,0.5
3,335985,Mumbai Indians,Royal Challengers Bengaluru,Royal Challengers Bengaluru,2008,Wankhede Stadium,Mumbai Indians,bat,165,166,...,1,0,1500.0,1484.0,16.0,0.5,0.0,0.5,0.5,0.5
4,335986,Sunrisers Hyderabad,Kolkata Knight Riders,Kolkata Knight Riders,2008,Eden Gardens,Sunrisers Hyderabad,bat,110,112,...,1,0,1500.0,1516.0,-16.0,0.5,1.0,0.5,-0.5,0.5


## 3. Pre-Match Model
Internal evaluation: train on seasons <= 2020, test on seasons > 2020.

**Caveat:** pre-match win prediction is close to random (AUC ~0.5) -- see `../docs/known_limitations.md`.

In [3]:
pre = train_pre_match_internal(match_df)
print(f"Climatology  Brier={pre['climatology']['brier']:.4f}  AUC={pre['climatology']['auc']:.4f}")
print(f"Cal. LR      Brier={pre['cal_lr']['brier']:.4f}  AUC={pre['cal_lr']['auc']:.4f}  "
      f"Acc={pre['cal_lr']['acc']:.4f}  BSS={pre['cal_lr']['bss']:.4f}  ECE={pre['cal_lr']['ece']:.4f}")
print(f"Cal. GBT     Brier={pre['cal_gbt']['brier']:.4f}  AUC={pre['cal_gbt']['auc']:.4f}  "
      f"Acc={pre['cal_gbt']['acc']:.4f}  BSS={pre['cal_gbt']['bss']:.4f}  ECE={pre['cal_gbt']['ece']:.4f}")

/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights 

Climatology  Brier=0.2508  AUC=0.5000
Cal. LR      Brier=0.2637  AUC=0.4527  Acc=0.4755  BSS=-0.0516  ECE=0.1092
Cal. GBT     Brier=0.2559  AUC=0.5035  Acc=0.4986  BSS=-0.0204  ECE=0.0654


/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library

## 4. Dynamic 2nd/1st-Innings Models

This is the **default win-probability path** -- verified AUC 0.878 (2nd innings) vs. project_hrishav's 0.807 on their own respective test sets.

In [4]:
df2 = build_dynamic_2nd(df, match_df)
df1 = build_dynamic_1st(df, match_df)
dyn = train_dynamic_internal(df2, df1)
print(f"Dynamic 2nd  Brier={dyn['dyn2']['brier']:.4f}  AUC={dyn['dyn2']['auc']:.4f}  "
      f"Acc={dyn['dyn2']['acc']:.4f}  ECE={dyn['dyn2']['ece']:.4f}")
print(f"Dynamic 1st  Brier={dyn['dyn1']['brier']:.4f}  AUC={dyn['dyn1']['auc']:.4f}  "
      f"Acc={dyn['dyn1']['acc']:.4f}")

/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/skl

/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/skl

/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: Ru

Dynamic 2nd  Brier=0.1445  AUC=0.8782  Acc=0.7918  ECE=0.0430
Dynamic 1st  Brier=0.2203  AUC=0.7022  Acc=0.6395


/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: Ru

## 5. Phase-Specific Evaluation (2nd innings)

Separate calibrated model per match phase (Powerplay/Middle/Death).

In [5]:
phases = phase_specific_eval(dyn['train2'], dyn['test2'], dyn['dsc2'])
for name, m in phases.items():
    print(f"{name:22s}  AUC={m['auc']:.4f}  Brier={m['brier']:.4f}  n={m['n']}")

/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/skl

/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: Ru

/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library

Powerplay (0-6 ov)      AUC=0.8210  Brier=0.1784  n=15098
Middle (6-15 ov)        AUC=0.8857  Brier=0.1381  n=18889
Death (15-20 ov)        AUC=0.9534  Brier=0.0836  n=6383


## 6. Score Regression Zoo

Saves the canonical score-prediction artefact to `../models/ipl_score_pipeline.pkl`.

**Caveat:** pre-match score R² is expected to be negative (worse than predicting the mean) -- this mirrors project_gagan's own finding and is not a bug.

In [6]:
score_metrics = train_score_zoo_and_save(df1, df2, match_df, SCORE_PIPELINE_PATH)
for stage_name, stage_metrics in score_metrics.items():
    print(f'{stage_name}:')
    for model_name, m in stage_metrics.items():
        print(f"  {model_name:15s} MAE={m['MAE']:.1f}  R2={m['R2']:.3f}")

/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/

/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/

pre_match:
  Linear          MAE=30.9  R2=-0.219
  Random Forest   MAE=32.6  R2=-0.338
  Gradient BT     MAE=32.8  R2=-0.350
  XGBoost         MAE=32.8  R2=-0.335
  SVR             MAE=32.8  R2=-0.362
inn1:
  Linear          MAE=18.7  R2=0.491
  Random Forest   MAE=19.1  R2=0.439
  Gradient BT     MAE=18.9  R2=0.452
  XGBoost         MAE=18.9  R2=0.447
  SVR             MAE=18.4  R2=0.503
inn2:
  Linear          MAE=19.6  R2=0.382
  Random Forest   MAE=14.0  R2=0.597
  Gradient BT     MAE=13.6  R2=0.618
  XGBoost         MAE=13.3  R2=0.631
  SVR             MAE=17.8  R2=0.441


/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


## 7. External 2026 Holdout Evaluation

Locked holdout, never used in any of the internal evaluation above. Retrains the pre-match model on the full 2008-2025 dataset first (cell 49 of project_gagan's original notebook) -- using the internal-only model here would understate real performance.

In [7]:
print('Retraining pre-match model on full 2008-2025 data...')
pre_model_full, sc_pre_full = retrain_pre_match_full(match_df)
ext = evaluate_2026_pre_match(match_df, pre_model_full, sc_pre_full, PM26_CSV)

print(f"Pre-match accuracy: {ext['accuracy']:.1%}  ({ext['k']}/{ext['n']})")
print(f"95% CI (Clopper-Pearson): [{ext['ci'][0]:.1%}, {ext['ci'][1]:.1%}]")
print(f"p-value vs 50/50: {ext['p_value']:.3f}")
print(f"Naive majority-class baseline: {ext['naive_baseline_acc']:.1%}")

/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/gnandeepboppudi/Library/Python/3.9/lib/python/site-packages/skl

Retraining pre-match model on full 2008-2025 data...
Pre-match accuracy: 64.9%  (48/74)
95% CI (Clopper-Pearson): [52.9%, 75.6%]
p-value vs 50/50: 0.011
Naive majority-class baseline: 63.5%


> **Note:** pre-match AUC is ~0.51 on internal LOSO (near-random) -- the 2026 accuracy figure above may reflect variance over 74 matches rather than genuine model skill. See `../docs/known_limitations.md`.

## 8. SHAP Explainability

Explains the calibrated Gradient Boosted Trees pre-match model's predict_proba output.

In [8]:
train_m = match_df[match_df['year'] <= 2020]
test_m = match_df[match_df['year'] > 2020]
X_tr = pre['sc'].transform(train_m[PRE_FEAT])
X_te = pre['sc'].transform(test_m[PRE_FEAT])

shap_result = shap_importance(pre['cal_gbt_model'], X_tr, X_te, PRE_FEAT)
for feat, val in shap_result['importance'].items():
    print(f'{feat:18s} mean|SHAP|={val:.4f}')

PermutationExplainer explainer: 100%|██████████| 80/80 [00:00<?, ?it/s]

PermutationExplainer explainer: 81it [00:10, 10.03s/it]                

elo_diff           mean|SHAP|=0.0439
h2h                mean|SHAP|=0.0177
form_diff          mean|SHAP|=0.0128
toss_field_first   mean|SHAP|=0.0007
toss_bat_first     mean|SHAP|=0.0004


## 9. 2026 Tournament Simulation

Actual vs. predicted points table, built from the pre-match model's picks.

In [9]:
ig26 = pd.read_csv(IG26_CSV)
table_actual = actual_points_table(ext['pm26'], ig26)
table_pred = predicted_points_table(ext['pre_df'])
cmp = compare_tables(table_actual, table_pred)

print(f"Actual table topper   : {cmp['actual_topper']}")
print(f"Predicted table topper: {cmp['predicted_topper']}  "
      f"({'CORRECT' if cmp['topper_correct'] else 'WRONG'})")
print(f"Top-4 overlap: {cmp['top4_overlap_count']}/4  {cmp['top4_overlap']}")
table_actual

Actual table topper   : Royal Challengers Bengaluru
Predicted table topper: Royal Challengers Bengaluru  (CORRECT)
Top-4 overlap: 3/4  ['Gujarat Titans', 'Punjab Kings', 'Royal Challengers Bengaluru']


,Team,P,W,L,Pts,NRR
1,Royal Challengers Bengaluru,16,11,5,22,1.064
2,Gujarat Titans,17,10,7,20,0.254
3,Sunrisers Hyderabad,15,9,6,18,0.334
4,Punjab Kings,14,8,6,16,0.304
5,Rajasthan Royals,16,8,8,16,0.265
6,Delhi Capitals,14,7,7,14,-0.660
7,Kolkata Knight Riders,14,6,8,12,-0.112
8,Chennai Super Kings,14,6,8,12,-0.339
9,Lucknow Super Giants,14,5,9,10,-0.762
10,Mumbai Indians,14,4,10,8,-0.561


## Summary

This is the recommended win-probability path for the merged project. For the secondary alternative model (causal Transformer, hrishav-derived, not recommended as the default -- see `../docs/known_limitations.md`), see `run_alt_transformer.ipynb`.